# Silver - Clientes e Atendentes

Limpeza, normalização e formatação das tabelas de atendentes e clientes a partir da camada Bronze.

- Entrada: bronze.tb_base_atendentes e bronze.tb_clientes (todas as colunas como string).
- Saída: tabelas de mesmo nome em silver, com tipos ajustados.

- Nenhuma linha é descartada: valor inválido vira null, nunca é inventado.
- ingestion_datetime é mantida para rastreabilidade.
- Chaves duplicadas interrompem a execução, duplicidades de negócio apenas geram alerta.

Mesmas constantes de catálogo e schemas dos notebooks anteriores, para manter o padrão estabelecido, combinado via docs do google. A Silver lê somente da Bronze, conforme a arquitetura de medalhão.

In [0]:
from pyspark.sql import functions as F  # Funções de transformação do PySpark

# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")

BRONZE_SCHEMA: vcredit_medallion.bronze
SILVER_SCHEMA: vcredit_medallion.silver


Funções reutilizadas nas tabelas de cliente e atendente, para aplicar a mesma regra em todas as colunas equivalentes.

- limpar_texto: remove espaços das pontas, colapsa espaços repetidos e converte vazio em null. O CSV usa string vazia para ausência, e null é a representação correta em Delta.
- para_inteiro: converte para inteiro só se o texto for numéric, caso contrário retorna null, sem erro de cast.
- para_data: aceita dd/MM/yyyy e yyyy-MM-dd, os dois formatos presentes nos dados, qualquer outro vira null.
- verificar e relatorio_invalidados: validações com log padronizado (OK/WARN/FAIL).

In [0]:
def limpar_texto(coluna): # Padroniza texto: trim, espaços únicos e vazio -> null
    c = F.trim(F.regexp_replace(coluna, r"\s+", " ")) # Colapsa espaços repetidos e remove das pontas
    return F.when(c == "", F.lit(None)).otherwise(c) # Vazio vira null


def para_inteiro(coluna): # Converte texto numérico em int
    c = limpar_texto(coluna) # Limpa antes de converter
    return F.when(c.rlike(r"^\d+$"), c.cast("int")) # Não numérico vira null


def para_data(coluna): # Converte texto em date aceitando os dois formatos encontrados
    c = F.trim(coluna) # Remove espaços das pontas
    return (
        F.when(c.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(c, "dd/MM/yyyy")) # Formato brasileiro
        .when(c.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(c, "yyyy-MM-dd")) # Formato ISO
    ) # Outros formatos viram null


def verificar(descricao, qtd, critico=False): # Loga o resultado de uma validação
    if qtd == 0:
        print(f"{OK} {descricao}: 0")
    elif critico:
        print(f"{FAIL} {descricao}: {qtd}")
        raise Exception(f"Validação crítica falhou: {descricao} ({qtd})") # Interrompe o notebook
    else:
        print(f"{WARN} {descricao}: {qtd}") # Alerta sem interromper


def relatorio_invalidados(df_bronze, df_silver, colunas): # Conta valores preenchidos na Bronze que viraram null
    b = df_bronze.agg(*[F.count(limpar_texto(F.col(c))).alias(c) for c in colunas]).first() # Preenchidos na Bronze
    s = df_silver.agg(*[F.count(F.col(c)).alias(c) for c in colunas]).first() # Preenchidos na Silver
    for c in colunas:
        verificar(f"{c}: valores preenchidos invalidados (viraram null)", b[c] - s[c])


def gravar_silver(df, df_bronze, nome_tabela): # Grava em Delta e valida a escrita
    tabela = f"{SILVER_SCHEMA}.{nome_tabela}"
    (
        df.write.format("delta")
        .mode("overwrite") # Lote único, como na Bronze
        .option("overwriteSchema", "true") # Acomoda mudanças de schema entre reexecuções
        .saveAsTable(tabela)
    )
    total_silver = spark.table(tabela).count()
    total_bronze = df_bronze.count()
    if total_silver == 0 or total_silver != total_bronze: # Tabela vazia ou linhas perdidas é erro
        print(f"{FAIL} {tabela}: {total_silver} linhas (Bronze: {total_bronze})")
        raise Exception(f"Contagem divergente em {tabela}")
    print(f"{OK} {tabela}: {total_silver} linhas")
    spark.table(tabela).printSchema() # Confere os tipos gravados
    display(spark.table(tabela).limit(5)) # Amostra para conferência visual

As duas tabelas são lidas como estão (strings). Nenhuma transformação nesta etapa.

In [0]:
df_atendentes_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_base_atendentes") # 25 linhas na Bronze (vide a execução)
df_clientes_bronze = spark.table(f"{BRONZE_SCHEMA}.tb_clientes") # 47.705 linhas na Bronze (vide a execução)

print(f"tb_base_atendentes: {df_atendentes_bronze.count()} linhas")
print(f"tb_clientes: {df_clientes_bronze.count()} linhas")

tb_base_atendentes: 25 linhas
tb_clientes: 47705 linhas


- id_atendente: convertido para inteiro.
- nome_atendente: apenas limpeza de espaços.
- nivel_atendimento: chega em 5 formas para só 2 níveis (1, Nível 1, 2, Nível 2, N2). Extrai o dígito e grava como inteiro (1 ou 2).
- data_admissao e data_desligamento: dd/MM/yyyy convertido para date. Desligamento vazio vira null (atendente em atividade).
- status_atendente: normalizado para minúsculas e sem espaços (ativo/inativo), mantido como texto.

In [0]:
df_atendentes = df_atendentes_bronze.select(
    para_inteiro(F.col("id_atendente")).alias("id_atendente"), # Identificador como inteiro
    limpar_texto(F.col("nome_atendente")).alias("nome_atendente"), # Remove espaços extras
    para_inteiro(F.regexp_extract(F.col("nivel_atendimento"), r"(\d+)", 1)).alias("nivel_atendimento"), # 'Nível 1'/'N2'/'2' -> 1 ou 2
    para_data(F.col("data_admissao")).alias("data_admissao"), # dd/MM/yyyy -> date
    para_data(F.col("data_desligamento")).alias("data_desligamento"), # Vazio -> null (ativo)
    F.lower(limpar_texto(F.col("status_atendente"))).alias("status_atendente"), # Minúsculas e sem espaços
    F.col("ingestion_datetime"), # Mantida para rastreabilidade
)

display(df_atendentes.orderBy("id_atendente"))

id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime
1,Marcela França,1,2019-04-18,2025-11-16,inativo,2026-10-25T06:45:46.016Z
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo,2026-10-25T06:45:46.016Z
3,William Marques,1,2019-05-28,2025-04-15,inativo,2026-10-25T06:45:46.016Z
4,Solange Silveira,1,2019-10-07,null,ativo,2026-10-25T06:45:46.016Z
5,Joaquim Barros,1,2020-12-06,2026-05-23,inativo,2026-10-25T06:45:46.016Z
6,Roberta Assunção,1,2021-10-29,null,ativo,2026-10-25T06:45:46.016Z
7,Luiza Galvão,1,2022-04-24,null,ativo,2026-10-25T06:45:46.016Z
8,Josefa Brito,1,2023-01-25,null,ativo,2026-10-25T06:45:46.016Z
9,Jaqueline Campos,1,2019-04-19,null,ativo,2026-10-25T06:45:46.016Z
10,Marcos Torres,1,2020-01-20,null,ativo,2026-10-25T06:45:46.016Z


- Chave id_atendente nula ou duplicada interrompe a execução (crítico).
- Alertas para: valores invalidados pela conversão, status fora de ativo/inativo, status incoerente com a data de desligamento, e desligamento anterior à admissão.

In [0]:
total = df_atendentes.count()

# Chave primária: nula ou duplicada é crítico
ids_validos = df_atendentes.where(F.col("id_atendente").isNotNull()).select("id_atendente").distinct().count()
verificar("id_atendente nulo ou duplicado", total - ids_validos, critico=True)

# Valores que existiam na Bronze e viraram null na conversão
relatorio_invalidados(df_atendentes_bronze, df_atendentes,
                      ["nome_atendente", "nivel_atendimento", "data_admissao", "data_desligamento", "status_atendente"])

# Domínio e coerência
verificar("status fora de ativo/inativo", df_atendentes.where(~F.col("status_atendente").isin("ativo", "inativo")).count())
verificar("status incoerente com data_desligamento",
          df_atendentes.where((F.col("status_atendente") == "inativo") != F.col("data_desligamento").isNotNull()).count()) # inativo deve ter desligamento e vice-versa
verificar("desligamento anterior à admissão",
          df_atendentes.where(F.col("data_desligamento") < F.col("data_admissao")).count())

✅ id_atendente nulo ou duplicado: 0
✅ nome_atendente: valores preenchidos invalidados (viraram null): 0
✅ nivel_atendimento: valores preenchidos invalidados (viraram null): 0
✅ data_admissao: valores preenchidos invalidados (viraram null): 0
✅ data_desligamento: valores preenchidos invalidados (viraram null): 0
✅ status_atendente: valores preenchidos invalidados (viraram null): 0
✅ status fora de ativo/inativo: 0
✅ status incoerente com data_desligamento: 0
✅ desligamento anterior à admissão: 0


Escrita em Delta (overwrite), com conferência de contagem contra a Bronze.

In [0]:
gravar_silver(df_atendentes, df_atendentes_bronze, "tb_base_atendentes")

✅ vcredit_medallion.silver.tb_base_atendentes: 25 linhas
root
 |-- id_atendente: integer (nullable = true)
 |-- nome_atendente: string (nullable = true)
 |-- nivel_atendimento: integer (nullable = true)
 |-- data_admissao: date (nullable = true)
 |-- data_desligamento: date (nullable = true)
 |-- status_atendente: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id_atendente,nome_atendente,nivel_atendimento,data_admissao,data_desligamento,status_atendente,ingestion_datetime
1,Marcela França,1,2019-04-18,2025-11-16,inativo,2026-10-25T06:45:46.016Z
2,Zilda Esteves,1,2023-09-12,2026-01-24,inativo,2026-10-25T06:45:46.016Z
3,William Marques,1,2019-05-28,2025-04-15,inativo,2026-10-25T06:45:46.016Z
4,Solange Silveira,1,2019-10-07,null,ativo,2026-10-25T06:45:46.016Z
5,Joaquim Barros,1,2020-12-06,2026-05-23,inativo,2026-10-25T06:45:46.016Z


- id_cliente: mantido como string. Tem sempre 11 dígitos e é um identificador.
- nome: apenas limpeza de espaços.
- email: três problemas encontrados e tratados em sequência:
  1. Espaços internos (ex.: ana . silva@gmail . com): todos os espaços são removidos, pois e-mail não os contém.
  2. Falta do @ (ex.: pessoaglobo.com): o @ é reinserido antes do domínio quando o texto termina em um dos domínios válidos observados nos próprios dados (DOMINIOS_EMAIL). Todos os 460 casos foram recuperados (vide a execução).
  3. Minúsculas e validação de formato: o que não passar no padrão vira null.
- regiao: trim e mapeamento para as 5 regiões do Brasil com grafia única (ex.: SUDESTE,  Sudeste -> Sudeste). Valor fora dessas 5 vira null.
- idade: inteiro. Valores fora de 18 a 100 viram null: há zeros (247) e valores de 121 a 199 (133), impossíveis. Vazios (459) também ficam null. Os limites de idades foram dados arbitrariamente e podem mudar mexendo em IDADE_MIN/IDADE_MAX. Observação: há um pico de clientes com 92 anos (algo que não é convecional, num caso real).
- data_cadastro: convertida para date, aceitando dd/MM/yyyy (maioria) e yyyy-MM-dd. Sem ambiguidade em /: a 1ª parte chega a 31 e a 2ª a no máximo 12, confirmando dd/MM.
- dia_vencimento_fatura: inteiro, aceito apenas de 1 a 31 (valores observados: 5, 10, 15, 20, 25, 28).

In [0]:
DOMINIOS_EMAIL = [ # Domínios válidos observados nos e-mails corretos
    "gmail.com", "hotmail.com", "outlook.com", "globo.com", "terra.com.br",
    "uol.com.br", "bol.com.br", "ig.com.br", "yahoo.com.br",
]
REGEX_EMAIL = r"^[a-z0-9._%+\-]+@[a-z0-9\-]+(\.[a-z]+)+$" # Padrão usuario@dominio.ext
IDADE_MIN, IDADE_MAX = 18, 100 # Premissa de negócio para idade válida
DIA_VENC_MIN, DIA_VENC_MAX = 1, 31 # Dia do mês válido

# E-mail: remove espaços e converte para minúsculas
email_base = F.lower(F.regexp_replace(F.col("email"), r"\s+", ""))

# E-mail sem '@': reinsere antes do domínio conhecido (domínios mais longos primeiro)
email_recuperado = F.lit(None).cast("string")
for dominio in sorted(DOMINIOS_EMAIL, key=len, reverse=True):
    email_recuperado = F.coalesce(
        email_recuperado,
        F.when(
            email_base.endswith(dominio) & (F.length(email_base) > len(dominio)), # Termina no domínio e sobra usuário
            F.concat(email_base.substr(F.lit(1), F.length(email_base) - F.lit(len(dominio))), F.lit("@"), F.lit(dominio)),
        ),
    )
email_corrigido = F.when(email_base.contains("@"), email_base).otherwise(email_recuperado) # Só recupera quando falta '@'
email_final = F.when(email_corrigido.rlike(REGEX_EMAIL), email_corrigido) # Formato inválido vira null

# Região: grafia única das 5 regiões
regiao_chave = F.lower(F.trim(F.col("regiao")))
regiao_final = (
    F.when(regiao_chave == "norte", "Norte")
    .when(regiao_chave == "nordeste", "Nordeste")
    .when(regiao_chave == "centro-oeste", "Centro-Oeste")
    .when(regiao_chave == "sudeste", "Sudeste")
    .when(regiao_chave == "sul", "Sul")
)  # Fora das 5 vira null

idade_num = para_inteiro(F.col("idade")) # Idade como inteiro antes da regra de faixa
dia_num = para_inteiro(F.col("dia_vencimento_fatura")) # Dia como inteiro antes da regra de faixa

df_clientes = df_clientes_bronze.select(
    limpar_texto(F.col("id_cliente")).alias("id_cliente"), # Mantido string (preserva zeros à esquerda)
    limpar_texto(F.col("nome")).alias("nome"), # Remove espaços extras
    email_final.alias("email"), # Limpo, recuperado e validado
    regiao_final.alias("regiao"), # Grafia padronizada
    F.when(idade_num.between(IDADE_MIN, IDADE_MAX), idade_num).alias("idade"), # Fora da faixa -> null
    para_data(F.col("data_cadastro")).alias("data_cadastro"), # Dois formatos -> date
    F.when(dia_num.between(DIA_VENC_MIN, DIA_VENC_MAX), dia_num).alias("dia_vencimento_fatura"), # Fora da faixa -> null
    F.col("ingestion_datetime"), # Mantida para rastreabilidade
)

display(df_clientes.limit(20))

id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura,ingestion_datetime
56324416706,S�rgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,2024-10-14,10,2026-10-25T06:46:11.911Z
97653229900,Leandro Magalh�es Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10,2026-10-25T06:46:11.911Z
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,2019-02-16,25,2026-10-25T06:46:11.911Z
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,2025-08-27,20,2026-10-25T06:46:11.911Z
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,2026-05-07,15,2026-10-25T06:46:11.911Z
11173485227,Renata Freitas Fernandes,renata96@yahoo.com.br,Sudeste,37,2025-01-26,5,2026-10-25T06:46:11.911Z
50855203771,Lucas Cardoso,lucascardoso@uol.com.br,Nordeste,38,2026-07-15,28,2026-10-25T06:46:11.911Z
72534560781,Camila Guimar�es Gomes,camila_gomes@uol.com.br,Nordeste,80,2025-09-03,25,2026-10-25T06:46:11.911Z
40409080977,Thiago Peixoto,thiago98@gmail.com,Sul,92,2023-06-21,25,2026-10-25T06:46:11.911Z
65936173686,Mariana Aparecida Esteves,esteves.mariana@ig.com.br,Norte,41,2025-05-31,15,2026-10-25T06:46:11.911Z


- Chave id_cliente nula ou duplicada interrompe a execução (crítico).
- Demais verificações geram apenas alerta: formato do id, valores invalidados pelo tratamento, data de cadastro futura, caracteres corrompidos em nome e cadastros duplicados com ids diferentes.
- Os cadastros duplicados (mesmos dados, id_cliente diferente) são mantidos: o id é a chave que liga aos chamados, e decidir se são a mesma pessoa é regra de negócio, não de limpeza.
- O alerta de caracteres corrompidos (�) existe porque o clientes.csv está em ISO-8859-1 e a Bronze o lê como UTF-8, o que já destrói os acentos na origem. Isso foi conversado no grupo do case e vai ser resolvido, e com isso vai arrumar esta camada silver, a partir da reexecução da bronze consertada e deste silver.

In [0]:
total = df_clientes.count()

# Chave primária: nula ou duplicada é crítico
ids_validos = df_clientes.where(F.col("id_cliente").isNotNull()).select("id_cliente").distinct().count()
verificar("id_cliente nulo ou duplicado", total - ids_validos, critico=True)
verificar("id_cliente fora do formato de 11 dígitos", df_clientes.where(~F.col("id_cliente").rlike(r"^\d{11}$")).count())

# Valores que existiam na Bronze e viraram null no tratamento
relatorio_invalidados(df_clientes_bronze, df_clientes,
                      ["nome", "email", "regiao", "idade", "data_cadastro", "dia_vencimento_fatura"])

# Datas e codificação
verificar("data_cadastro futura", df_clientes.where(F.col("data_cadastro") > F.current_date()).count())
verificar("nome com caractere corrompido (encoding da Bronze)", df_clientes.where(F.col("nome").contains("\ufffd")).count())

# Cadastros com todos os dados iguais, exceto id_cliente (mantidos)
colunas_negocio = [c for c in df_clientes.columns if c not in ("id_cliente", "ingestion_datetime")]
verificar("cadastros duplicados com id_cliente diferente (mantidos)",
          total - df_clientes.dropDuplicates(colunas_negocio).count())

✅ id_cliente nulo ou duplicado: 0
✅ id_cliente fora do formato de 11 dígitos: 0
✅ nome: valores preenchidos invalidados (viraram null): 0
✅ email: valores preenchidos invalidados (viraram null): 0
✅ regiao: valores preenchidos invalidados (viraram null): 0
⚠️ idade: valores preenchidos invalidados (viraram null): 380
✅ data_cadastro: valores preenchidos invalidados (viraram null): 0
✅ dia_vencimento_fatura: valores preenchidos invalidados (viraram null): 0
✅ data_cadastro futura: 0
⚠️ nome com caractere corrompido (encoding da Bronze): 12506
⚠️ cadastros duplicados com id_cliente diferente (mantidos): 705


Escrita em Delta (overwrite), com conferência de contagem contra a Bronze.

In [0]:
gravar_silver(df_clientes, df_clientes_bronze, "tb_clientes")

✅ vcredit_medallion.silver.tb_clientes: 47705 linhas
root
 |-- id_cliente: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- dia_vencimento_fatura: integer (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura,ingestion_datetime
56324416706,S�rgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,2024-10-14,10,2026-10-25T06:46:11.911Z
97653229900,Leandro Magalh�es Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10,2026-10-25T06:46:11.911Z
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,2019-02-16,25,2026-10-25T06:46:11.911Z
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,2025-08-27,20,2026-10-25T06:46:11.911Z
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,2026-05-07,15,2026-10-25T06:46:11.911Z
